In [1]:
import os
import pandas as pd
import joblib
from datetime import datetime
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from src.config import *
from src.utils import *
from src.modeling import prepare_model_input, get_optimal_model_params
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline


start_time = datetime.now()

In [2]:
import xgboost as xgb

def is_gpu_available():
    try:
        params = {
            "tree_method": "gpu_hist",
            "predictor": "gpu_predictor",
            "nthread": 1,
        }
        dmatrix = xgb.DMatrix(data=[[1], [2]], label=[0, 1])
        xgb.train(params, dmatrix, num_boost_round=1)
        return True
    except xgb.core.XGBoostError:
        return False

USE_GPU = is_gpu_available()


In [3]:
n_jobs = min(os.cpu_count() - 2, 4)
n_jobs

4

In [4]:



# Charger les données
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
df = pd.read_csv(dataset_path)

# Préparation des données
# target = 'IS_WIN'
# drop_cols = COLS_TO_DROP_TARGET_IS_WIN + COLS_ODDS
# features = [col for col in df.columns if col not in drop_cols + [target]]
# df = df.dropna(subset=features)

target = 'IS_WIN'
X = prepare_model_input(df, target=target)
y = df.loc[X.index, target]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)




# Définis tes modèles de base
estimators = [
    ('rf', RandomForestClassifier(**get_optimal_model_params('rf', use_gpu=USE_GPU, max_cpu_jobs=n_jobs))),
    ('lgbm', LGBMClassifier(**get_optimal_model_params('lgbm', use_gpu=USE_GPU, max_cpu_jobs=n_jobs))),
    ('xgb', XGBClassifier(**get_optimal_model_params('xgb', use_gpu=False, max_cpu_jobs=n_jobs))),
    ('cat', CatBoostClassifier(**get_optimal_model_params('cat', use_gpu=USE_GPU, max_cpu_jobs=n_jobs))),
    ('hgb', HistGradientBoostingClassifier(max_iter=200, random_state=42)),  # Gère déjà les NaN
    ('lr', make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver='liblinear', penalty='l2', n_jobs=n_jobs)
    )),
    # Ajoute d'autres modèles si besoin
]


# Meta-model (peut être LogisticRegression, simple et efficace)
meta_model = LogisticRegression(solver='lbfgs', max_iter=5000)

# Création du stacking
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    passthrough=False,
    cv=5,
    n_jobs=1,
    verbose=2
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', stack)
])

# Entraînement
pipeline.fit(X_train, y_train)

# Évaluation
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

# Sauvegarde
today = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

os.makedirs(DATA_MODELS_DIR, exist_ok=True)

stacking_model_path = os.path.join(DATA_MODELS_DIR, f"stacking_model_target_iswin_{today}.joblib")
joblib.dump(pipeline,stacking_model_path)
print(f"Model saved in {stacking_model_path}")

end_time = datetime.now()
print(f"Total execution time: {end_time - start_time}")



[LightGBM] [Info] Number of positive: 24251, number of negative: 26510
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.336140 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 351438
[LightGBM] [Info] Number of data points in the train set: 50761, number of used features: 1451
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.477749 -> initscore=-0.089064
[LightGBM] [Info] Start training from score -0.089064


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [15:30:31] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  7.8min finished


[LightGBM] [Info] Number of positive: 19400, number of negative: 21208
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.294202 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 351390
[LightGBM] [Info] Number of data points in the train set: 40608, number of used features: 1451
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.477738 -> initscore=-0.089105
[LightGBM] [Info] Start training from score -0.089105


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 19401, number of negative: 21208
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.301909 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 351385
[LightGBM] [Info] Number of data points in the train set: 40609, number of used features: 1451
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.477751 -> initscore=-0.089054
[LightGBM] [Info] Start training from score -0.089054


...

[LightGBM] [Info] Number of positive: 19401, number of negative: 21208
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.318354 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 351387
[LightGBM] [Info] Number of data points in the train set: 40609, number of used features: 1451
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.477751 -> initscore=-0.089054
[LightGBM] [Info] Start training from score -0.089054


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 19401, number of negative: 21208
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.344096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 351388
[LightGBM] [Info] Number of data points in the train set: 40609, number of used features: 1451
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.477751 -> initscore=-0.089054
[LightGBM] [Info] Start training from score -0.089054


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  1.9min finished
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [15:47:15] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [15:48:04] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [15:48:52] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder

ROC AUC: 0.7428693818219271
Model saved in data/models/stacking_model_target_iswin_2025-11-01_16-14-18.joblib
Total execution time: 0:46:28.798077


In [5]:
# import os
# import pandas as pd
# import joblib
# from datetime import datetime
# from pathlib import Path

# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import roc_auc_score
# from src.config import *
# from src.utils import get_latest_file
# from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

# from sklearn.model_selection import cross_val_score
# from sklearn.pipeline import make_pipeline

# from sklearn.ensemble import RandomForestRegressor, StackingRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
# from sklearn.linear_model import LinearRegression
# from lightgbm import LGBMRegressor
# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor
# from sklearn.neural_network import MLPRegressor
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.metrics import mean_squared_error, r2_score

# # Charger les données
# dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
# df = pd.read_csv(dataset_path)

# # Préparation des données
# target = 'POINT_DIFF'
# drop_cols = COLS_TO_DROP_TARGET_POINT_DIFF
# features = [col for col in df.columns if col not in drop_cols + [target]]
# df = df.dropna(subset=features)

# X = df[features]
# y = df[target]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Définis tes modèles de base
# estimators = [
#     ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
#     ('lgbm', LGBMRegressor(n_estimators=150, num_leaves=64, random_state=42, n_jobs=-1)),
#     ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.7, colsample_bytree=0.7, random_state=42, n_jobs=-1)),
#     ('cat', CatBoostRegressor(n_estimators=200, learning_rate=0.05, depth=6, rsm=0.8, verbose=0, random_state=42)),
#     ('hgb', HistGradientBoostingRegressor(max_iter=200, random_state=42)),
#     ('lr', make_pipeline(StandardScaler(), LinearRegression())),
#     ('mlp', make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42))),
#     ('et', ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
#     ('knn', make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15, n_jobs=-1)))
# ]

# # estimators_gpu = [
# #     ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),  # CPU only
# #     ('lgbm', LGBMRegressor(n_estimators=150, num_leaves=64, device='gpu', random_state=42, n_jobs=-1)),  # GPU
# #     ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
# #                          subsample=0.7, colsample_bytree=0.7, tree_method='gpu_hist',
# #                          random_state=42, n_jobs=-1)),  # GPU
# #     ('cat', CatBoostRegressor(n_estimators=200, learning_rate=0.05, depth=6,
# #                               verbose=0, random_state=42, task_type='GPU', devices='0')),  # GPU
# #     ('hgb', HistGradientBoostingRegressor(max_iter=200, random_state=42)),  # CPU only
# #     ('lr', make_pipeline(StandardScaler(), LinearRegression())),  # CPU
# #     ('mlp', make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42))),  # CPU
# #     ('et', ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1)),  # CPU
# #     ('knn', make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15, n_jobs=-1)))  # CPU
# # ]


# meta_model = LinearRegression()

# stack = StackingRegressor(
#     estimators=estimators,
#     final_estimator=meta_model,
#     passthrough=False,
#     cv=5,
#     n_jobs=-1,
#     verbose=2
# )


# pipeline = Pipeline([
#     ('scaler', StandardScaler()),
#     ('model', stack)
# ])

# # Entraînement
# pipeline.fit(X_train, y_train)

# # Évaluation
# y_pred = pipeline.predict(X_test)
# print("MSE:", mean_squared_error(y_test, y_pred))
# print("R² Score:", r2_score(y_test, y_pred))

# # Sauvegarde
# today = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# os.makedirs(DATA_MODELS_DIR, exist_ok=True)

# stacking_model_path = os.path.join(DATA_MODELS_DIR, f"stacking_model_target_pointdiff_{today}.joblib")
# joblib.dump(pipeline,stacking_model_path)
# print(f"Model saved in {stacking_model_path}")
